# Zhu-Net with different Database Partition

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Libraries

In [ ]:
from scipy import misc, ndimage, signal
from sklearn.model_selection  import train_test_split
import numpy
import numpy as np
import random
import ntpath
import os
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from keras import optimizers 
from keras import regularizers
import tensorflow as tf
import cv2
from keras import backend as K
from time import time
import time as tm
import datetime
from operator import itemgetter
import glob
from skimage.util.shape import view_as_blocks
from keras.utils import np_utils
from keras.utils import to_categorical

## 30 SRM filters for preprocessing and the activation function

In [ ]:
################################################## 30 SRM FILTERS
srm_weights = np.load('SRM_Kernels.npy') 
biasSRM=numpy.ones(30)
print (srm_weights.shape)
################################################## TLU ACTIVATION FUNCTION
T3 = 3;
def Tanh3(x):
    tanh3 = K.tanh(x)*T3
    return tanh3
##################################################

## Zhu-Net architecture

In [ ]:
def Zhu_Net(img_size=256):
    
    #Inputs
    inputs = tf.keras.Input(shape=(img_size,img_size,1),name="input_1") 
    
    #Layer 1
    layers = tf.keras.layers.Conv2D(30, (5,5), weights=[srm_weights,biasSRM], strides=(1,1), trainable=False, activation=Tanh3, use_bias=True)(inputs)
    layer1 = tf.keras.layers.BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)
    
    #Layer 2
    layer1 = tf.keras.layers.SpatialDropout2D(rate=0.1)(layer1)
    layer23 = tf.keras.layers.SeparableConv2D(30,(3,3),activation="relu",depth_multiplier=3,padding="same")(layer1) 
    layer23 = tf.keras.layers.Lambda(tf.keras.backend.abs)(layer23)
    layer23 = tf.keras.layers.BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layer23)
    
    #Layer 3
    layer23 = tf.keras.layers.SeparableConv2D(30,(3,3),activation="relu",depth_multiplier=3,padding="same")(layer23)
    layer23 = tf.keras.layers.Lambda(tf.keras.backend.abs)(layer23)
    layer23 = tf.keras.layers.BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layer23)
    
    #Shorcut
    layers= tf.keras.layers.Add()([layer23, layer1])
    
    #Layer 4
    layers = tf.keras.layers.Conv2D(32, (3,3), strides=(1,1), activation="relu", kernel_initializer='glorot_normal', padding='same')(layers) 
    layers = tf.keras.layers.Lambda(tf.keras.backend.abs)(layers)
    layers = tf.keras.layers.BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)
    layers = tf.keras.layers.AveragePooling2D((5,5), strides= (2,2),padding="same")(layers)
   
    #Layer 5
    layers = tf.keras.layers.Conv2D(32, (3,3), strides=(1,1), activation="relu", kernel_initializer='glorot_normal',padding="same")(layers) 
    layers = tf.keras.layers.Lambda(tf.keras.backend.abs)(layers)
    layers = tf.keras.layers.BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)
    layers = tf.keras.layers.AveragePooling2D((5,5), strides= (2,2),padding="same")(layers)
    
    #Layer 6
    layers = tf.keras.layers.Conv2D(64, (3,3), strides=(1,1), activation="relu", kernel_initializer='glorot_normal',padding="same")(layers)
    layers = tf.keras.layers.Lambda(tf.keras.backend.abs)(layers)
    layers = tf.keras.layers.BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)
    layers = tf.keras.layers.AveragePooling2D((5,5), strides= (2,2),padding="same")(layers)
    
    #Layer 7
    layers = tf.keras.layers.Conv2D(128, (5,5), strides=(1,1), activation="relu", kernel_initializer='glorot_normal',padding="same")(layers)
    layers = tf.keras.layers.Lambda(tf.keras.backend.abs)(layers)
    layers = tf.keras.layers.BatchNormalization(momentum=0.2, epsilon=0.001, center=True, scale=False, trainable=True, fused=None, renorm=False, renorm_clipping=None, renorm_momentum=0.4, adjustment=None)(layers)
    
    #Layer 8
    layers = tf.keras.layers.GlobalAveragePooling2D(data_format="channels_last")(layers)
    
    
    #Layer 9, FC, Softmax
    layers = tf.keras.layers.Dense(128,activation="relu")(layers)
    layers = tf.keras.layers.Dropout(0.2)(layers)
    layers = tf.keras.layers.Dense(64 ,activation="relu")(layers)
    layers = tf.keras.layers.Dropout(0.2)(layers)
    layers = tf.keras.layers.Dense(32 ,activation="relu")(layers)
    
    #Softmax
    predictions = tf.keras.layers.Dense(2, activation="softmax", name="output_1")(layers)
    
    #Model generation
    model = tf.keras.Model(inputs = inputs, outputs=predictions)
    
    #Optimizer
    optimizer=tf.keras.optimizers.RMSprop(lr=0.001, rho=0.9) 
    
    #Compilator
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    print ("Zhu-net model generated")
    return model

## Defining different functions to work with the architecture

In [ ]:
def train(model, X_train, y_train, X_valid, y_valid, X_test, y_test, batch_size, epochs, model_name=""):
    start_time = tm.time()
    log_dir=path_log_base+"/"+model_name+"_"+str(datetime.datetime.now().isoformat()[:19].replace("T", "_").replace(":","-"))
    tensorboard = tf.keras.callbacks.TensorBoard(log_dir, histogram_freq=1)
    filepath = log_dir+"/saved-model-{epoch:03d}-{val_accuracy:.4f}.hdf5"
    checkpoint = tf.keras.callbacks.ModelCheckpoint(filepath, monitor='val_accuracy', save_best_only=False, mode='max')
    model.reset_states()
    
    #VALORES EN TRAIN TEST Y VALIDACIÓN INICIALES, GRÁFICOS
    global lossTEST
    global accuracyTEST
    global lossTRAIN
    global accuracyTRAIN
    global lossVALID
    global accuracyVALID
    lossTEST,accuracyTEST   = model.evaluate(X_test, y_test,verbose=None)
    lossTRAIN,accuracyTRAIN = model.evaluate(X_train, y_train,verbose=None)
    lossVALID,accuracyVALID = model.evaluate(X_valid, y_valid,verbose=None)

    global history
    global model_Name
    global log_Dir
    model_Name = model_name
    log_Dir = log_dir
    
    history=model.fit(X_train, y_train, epochs=epochs, 
                      callbacks=[tensorboard,checkpoint], 
                      batch_size=batch_size,validation_data=(X_valid, y_valid),verbose=1)
    
    metrics = model.evaluate(X_test, y_test, verbose=0)
     
    TIME = tm.time() - start_time
    print("Time "+model_name+" = %s [seconds]" % TIME)
    
    print("\n")
    print(log_dir)
    return {k:v for k,v in zip (model.metrics_names, metrics)}

In [ ]:
def Final_Results_Test(model,PATH_trained_models):
    global AccTest
    global LossTest
    AccTest = []
    LossTest= [] 
    B_accuracy = 0 #B --> Best
    for filename in sorted(os.listdir(PATH_trained_models)):
      if filename != ('train') and filename != ('validation'):
        print(filename)
        model.load_weights(PATH_trained_models+'/'+filename)
        loss,accuracy = model.evaluate(X_test, y_test,verbose=0)
        print(f'Loss={loss:.4f} y Accuracy={accuracy:0.4f}'+'\n') 
        BandAccTest  = accuracy
        BandLossTest = loss
        AccTest.append(BandAccTest)    #Valores de la precisión en Test, para graficar junto a valid y train
        LossTest.append(BandLossTest)  #Valores de la perdida en Test, para graficar junto a valid y train
        if accuracy > B_accuracy:
          B_accuracy = accuracy
          B_loss = loss
          B_name = filename
    print("\n\nBest")
    print(B_name)
    print(f'Loss={B_loss:.4f} y Accuracy={B_accuracy:0.4f}'+'\n')

In [ ]:
from operator import itemgetter
def graphics(history, AccTest, LossTest, log_Dir, model_Name, lossTEST, lossTRAIN, lossVALID, accuracyTEST, accuracyTRAIN, accuracyVALID):
    numbers=AccTest
    numbers_sort = sorted(enumerate(numbers), key=itemgetter(1),  reverse=True)
    for i in range(int(len(numbers)*(0.05))): #5% Del total de las épocas
        index, value = numbers_sort[i]
        print("Test Accuracy {}, Época:{}\n".format(value, index+1))
    
    print("")
    
    numbers=history.history['accuracy']
    numbers_sort = sorted(enumerate(numbers), key=itemgetter(1),  reverse=True)
    for i in range(int(len(numbers)*(0.05))): #5% Del total de las épocas
        index, value = numbers_sort[i]
        print("Train Accuracy {}, Época:{}\n".format(value, index+1))
    
    print("")
    
    numbers=history.history['val_accuracy']
    numbers_sort = sorted(enumerate(numbers), key=itemgetter(1),  reverse=True)
    for i in range(int(len(numbers)*(0.05))): #5% Del total de las épocas
        index, value = numbers_sort[i]
        print("Validation Accuracy {}, Época:{}\n".format(value, index+1))

    with plt.style.context('seaborn-white'):
        plt.figure(figsize=(10, 10))
        #Plot training & validation accuracy values
        plt.plot(np.concatenate([np.array([accuracyTRAIN]),np.array(history.history['accuracy'])],axis=0))
        plt.plot(np.concatenate([np.array([accuracyVALID]),np.array(history.history['val_accuracy'])],axis=0))
        plt.plot(np.concatenate([np.array([accuracyTEST]),np.array(AccTest)],axis=0)) #Test
        plt.title('Accuracy Vs Epoch')
        plt.ylabel('Accuracy')
        plt.xlabel('Epoch')
        plt.legend(['Train', 'Validation', 'Test'], loc='upper left')
        plt.grid('on')
        plt.savefig(path_img_base+'/Accuracy_Zhu_Net_'+model_Name+'.eps', format='eps')
        plt.savefig(path_img_base+'/Accuracy_Zhu_Net_'+model_Name+'.svg', format='svg')
        plt.savefig(path_img_base+'/Accuracy_Zhu_Net_'+model_Name+'.pdf', format='pdf')     
        plt.show()
        
        plt.figure(figsize=(10, 10))
        #Plot training & validation loss values
        plt.plot(np.concatenate([np.array([lossTRAIN]),np.array(history.history['loss'])],axis=0))
        plt.plot(np.concatenate([np.array([lossVALID]),np.array(history.history['val_loss'])],axis=0))
        plt.plot(np.concatenate([np.array([lossTEST]),np.array(LossTest)],axis=0)) #Test
        plt.title('Loss Vs Epoch')
        plt.ylabel('Loss')
        plt.xlabel('Epoch')
        plt.legend(['Train', 'Validation', 'Test'], loc='upper left')
        plt.grid('on')
        plt.savefig(path_img_base+'/Loss_Zhu_Net_'+model_Name+'.eps', format='eps')
        plt.savefig(path_img_base+'/Loss_Zhu_Net_'+model_Name+'.svg', format='svg')
        plt.savefig(path_img_base+'/Loss_Zhu_Net_'+model_Name+'.pdf', format='pdf') 

In [ ]:
def top_models(AccTest,AccTrain,AccValid):
    numbers=AccTest
    numbers_sort = sorted(enumerate(numbers), key=itemgetter(1),  reverse=True)
    for i in range(int(len(numbers)*(0.05))): #5% total epochs
        index, value = numbers_sort[i]
        print("Test Accuracy {}, epoch:{}\n".format(value, index+1))
    
    print("")
    
    numbers=AccTrain
    numbers_sort = sorted(enumerate(numbers), key=itemgetter(1),  reverse=True)
    for i in range(int(len(numbers)*(0.05))): #5% total epochs
        index, value = numbers_sort[i]
        print("Train Accuracy {}, epoch:{}\n".format(value, index+1))
    
    print("")
    
    numbers=AccValid
    numbers_sort = sorted(enumerate(numbers), key=itemgetter(1),  reverse=True)
    for i in range(int(len(numbers)*(0.05))): #5% total epochs
        index, value = numbers_sort[i]
        print("Validation Accuracy {}, epoch:{}\n".format(value, index+1))

## Working with BOSSbase 1.01 WOW y PAYLOAD = 0.4bpp

In the README, there is a link to download the databases we use for the work. There is BOSSbase 1.01, cover images and stego. The steganographic algorithms used in the paper are WOW and S-UNIWARD, with a payload of 0.4bpp.

Note: If you want to train the algorithm with S-UNIWARD 0.4 bpp, change "PATH04_WOW1" and  "base_name".

# Database Partition

Choose any of the Database partition and train model. 

Note: Any of the nine "CNN Input" can be used to train your model, you can also download the databases in PGM format through a link found in the README.

## Training image pairs(4000), Validation image pairs(1000), and Test image pairs(5000)

In [ ]:
COVER_DIR = "cover256gris"

# 5 familias de stego
STEGO_DIRS = {
    "Troyano":        "jslamegris256",
    "Gusanos":        "beagle",
    "Apt":            "stuxnet",
    "Puerta_trasera": "linuxchapros256gris",
    "Ransomware":     "thanos256gris",
}

# Clases: cover=0, familia1=1, ..., familia5=5
CLASS_NAMES = ["cover"] + list(STEGO_DIRS.keys())
NUM_CLASSES  = len(CLASS_NAMES)  # 6
print("Clases:", CLASS_NAMES)

IMG_SIZE   = 256
BATCH_SIZE = 32
EPOCHS     = 20
TEST_RATIO  = 0.50
VAL_RATIO   = 0.20   # fracción del train que va a validación
SEED       = 42

# ======================
# SRM
# ======================
srm_weights = np.load("filtro/SRM_Kernels1.npy")
biasSRM     = np.ones(30)

# ======================
# UTILS
# ======================
def get_id(f):
    return os.path.splitext(f)[0].split("_")[0]

# ======================
# DATA SPLIT (PAIR SAFE)
# ======================
cover_map = {get_id(f): f for f in os.listdir(COVER_DIR)}

all_samples = []
for label_idx, (family_name, stego_dir) in enumerate(STEGO_DIRS.items(), start=1):
    stego_map  = {get_id(f): f for f in os.listdir(stego_dir)}
    common_ids = sorted(set(cover_map.keys()) & set(stego_map.keys()))
    print(f"  [{family_name}] pares válidos: {len(common_ids)}")
    for img_id in common_ids:
        all_samples.append({
            "cover_path":  os.path.join(COVER_DIR, cover_map[img_id]),
            "stego_path":  os.path.join(stego_dir, stego_map[img_id]),
            "stego_label": label_idx
        })

print(f"\nTotal pares (todas las familias): {len(all_samples)}")

np.random.seed(SEED)
indices = np.arange(len(all_samples))
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_RATIO, random_state=SEED, shuffle=True
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=VAL_RATIO, random_state=SEED, shuffle=True
)

train_samples = [all_samples[i] for i in train_idx]
val_samples   = [all_samples[i] for i in val_idx]
test_samples  = [all_samples[i] for i in test_idx]

print(f"Train: {len(train_samples)} pares → {len(train_samples)*2} imágenes")
print(f"Val:   {len(val_samples)} pares → {len(val_samples)*2} imágenes")
print(f"Test:  {len(test_samples)} pares → {len(test_samples)*2} imágenes")

# ======================
# SEQUENCE MULTICLASE
# ======================
class StegoSequence(tf.keras.utils.Sequence):
    def __init__(self, samples, batch_size, num_classes):
        self.samples     = samples
        self.batch       = batch_size
        self.num_classes = num_classes
        self.indices     = np.arange(len(self.samples))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.samples) * 2 / self.batch))

    def __getitem__(self, idx):
        half      = self.batch // 2
        batch_ids = self.indices[idx * half:(idx + 1) * half]
        X = np.zeros((len(batch_ids) * 2, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
        y = np.zeros((len(batch_ids) * 2, self.num_classes),       dtype=np.float32)
        j = 0
        for i in batch_ids:
            sample = self.samples[i]
            # COVER
            img = cv2.imread(sample["cover_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['cover_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(0, num_classes=self.num_classes)
            j += 1
            # STEGO
            img = cv2.imread(sample["stego_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['stego_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(sample["stego_label"], num_classes=self.num_classes)
            j += 1
        return X, y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_seq = StegoSequence(train_samples, BATCH_SIZE, NUM_CLASSES)
val_seq   = StegoSequence(val_samples,   BATCH_SIZE, NUM_CLASSES)
test_seq  = StegoSequence(test_samples,  BATCH_SIZE, NUM_CLASSES)

# ======================
# SANITY CHECK
# ======================
print("\n[CHECK] Probando un batch del Sequence...")
X_dbg, y_dbg = train_seq[0]
print("X batch shape:", X_dbg.shape)
print("y batch shape:", y_dbg.shape)
print("y sample (primeras 6):", y_dbg[:6])
print("[CHECK OK] Loader funcionando correctamente\n")

## Training image pairs(2500), Validation image pairs(2500), and Test image pairs(5000)

In [ ]:
COVER_DIR = "cover256gris"

# 5 familias de stego
STEGO_DIRS = {
    "Troyano":        "jslamegris256",
    "Gusanos":        "beagle",
    "Apt":            "stuxnet",
    "Puerta_trasera": "linuxchapros256gris",
    "Ransomware":     "thanos256gris",
}

# Clases: cover=0, familia1=1, ..., familia5=5
CLASS_NAMES = ["cover"] + list(STEGO_DIRS.keys())
NUM_CLASSES  = len(CLASS_NAMES)  # 6
print("Clases:", CLASS_NAMES)

IMG_SIZE   = 256
BATCH_SIZE = 32
EPOCHS     = 20
TEST_RATIO  = 0.50
VAL_RATIO   = 0.50   # fracción del train que va a validación
SEED       = 42

# ======================
# SRM
# ======================
srm_weights = np.load("filtro/SRM_Kernels1.npy")
biasSRM     = np.ones(30)

# ======================
# UTILS
# ======================
def get_id(f):
    return os.path.splitext(f)[0].split("_")[0]

# ======================
# DATA SPLIT (PAIR SAFE)
# ======================
cover_map = {get_id(f): f for f in os.listdir(COVER_DIR)}

all_samples = []
for label_idx, (family_name, stego_dir) in enumerate(STEGO_DIRS.items(), start=1):
    stego_map  = {get_id(f): f for f in os.listdir(stego_dir)}
    common_ids = sorted(set(cover_map.keys()) & set(stego_map.keys()))
    print(f"  [{family_name}] pares válidos: {len(common_ids)}")
    for img_id in common_ids:
        all_samples.append({
            "cover_path":  os.path.join(COVER_DIR, cover_map[img_id]),
            "stego_path":  os.path.join(stego_dir, stego_map[img_id]),
            "stego_label": label_idx
        })

print(f"\nTotal pares (todas las familias): {len(all_samples)}")

np.random.seed(SEED)
indices = np.arange(len(all_samples))
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_RATIO, random_state=SEED, shuffle=True
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=VAL_RATIO, random_state=SEED, shuffle=True
)

train_samples = [all_samples[i] for i in train_idx]
val_samples   = [all_samples[i] for i in val_idx]
test_samples  = [all_samples[i] for i in test_idx]

print(f"Train: {len(train_samples)} pares → {len(train_samples)*2} imágenes")
print(f"Val:   {len(val_samples)} pares → {len(val_samples)*2} imágenes")
print(f"Test:  {len(test_samples)} pares → {len(test_samples)*2} imágenes")

# ======================
# SEQUENCE MULTICLASE
# ======================
class StegoSequence(tf.keras.utils.Sequence):
    def __init__(self, samples, batch_size, num_classes):
        self.samples     = samples
        self.batch       = batch_size
        self.num_classes = num_classes
        self.indices     = np.arange(len(self.samples))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.samples) * 2 / self.batch))

    def __getitem__(self, idx):
        half      = self.batch // 2
        batch_ids = self.indices[idx * half:(idx + 1) * half]
        X = np.zeros((len(batch_ids) * 2, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
        y = np.zeros((len(batch_ids) * 2, self.num_classes),       dtype=np.float32)
        j = 0
        for i in batch_ids:
            sample = self.samples[i]
            # COVER
            img = cv2.imread(sample["cover_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['cover_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(0, num_classes=self.num_classes)
            j += 1
            # STEGO
            img = cv2.imread(sample["stego_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['stego_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(sample["stego_label"], num_classes=self.num_classes)
            j += 1
        return X, y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_seq = StegoSequence(train_samples, BATCH_SIZE, NUM_CLASSES)
val_seq   = StegoSequence(val_samples,   BATCH_SIZE, NUM_CLASSES)
test_seq  = StegoSequence(test_samples,  BATCH_SIZE, NUM_CLASSES)

# ======================
# SANITY CHECK
# ======================
print("\n[CHECK] Probando un batch del Sequence...")
X_dbg, y_dbg = train_seq[0]
print("X batch shape:", X_dbg.shape)
print("y batch shape:", y_dbg.shape)
print("y sample (primeras 6):", y_dbg[:6])
print("[CHECK OK] Loader funcionando correctamente\n")

## Training image pairs(4000), Validation image pairs(3000), and Test image pairs(3000)

In [ ]:
COVER_DIR = "cover256gris"

# 5 familias de stego
STEGO_DIRS = {
    "Troyano":        "jslamegris256",
    "Gusanos":        "beagle",
    "Apt":            "stuxnet",
    "Puerta_trasera": "linuxchapros256gris",
    "Ransomware":     "thanos256gris",
}

# Clases: cover=0, familia1=1, ..., familia5=5
CLASS_NAMES = ["cover"] + list(STEGO_DIRS.keys())
NUM_CLASSES  = len(CLASS_NAMES)  # 6
print("Clases:", CLASS_NAMES)

IMG_SIZE   = 256
BATCH_SIZE = 32
EPOCHS     = 20
TEST_RATIO  = 0.30
VAL_RATIO   = 0.43   # fracción del train que va a validación
SEED       = 42

# ======================
# SRM
# ======================
srm_weights = np.load("filtro/SRM_Kernels1.npy")
biasSRM     = np.ones(30)

# ======================
# UTILS
# ======================
def get_id(f):
    return os.path.splitext(f)[0].split("_")[0]

# ======================
# DATA SPLIT (PAIR SAFE)
# ======================
cover_map = {get_id(f): f for f in os.listdir(COVER_DIR)}

all_samples = []
for label_idx, (family_name, stego_dir) in enumerate(STEGO_DIRS.items(), start=1):
    stego_map  = {get_id(f): f for f in os.listdir(stego_dir)}
    common_ids = sorted(set(cover_map.keys()) & set(stego_map.keys()))
    print(f"  [{family_name}] pares válidos: {len(common_ids)}")
    for img_id in common_ids:
        all_samples.append({
            "cover_path":  os.path.join(COVER_DIR, cover_map[img_id]),
            "stego_path":  os.path.join(stego_dir, stego_map[img_id]),
            "stego_label": label_idx
        })

print(f"\nTotal pares (todas las familias): {len(all_samples)}")

np.random.seed(SEED)
indices = np.arange(len(all_samples))
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_RATIO, random_state=SEED, shuffle=True
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=VAL_RATIO, random_state=SEED, shuffle=True
)

train_samples = [all_samples[i] for i in train_idx]
val_samples   = [all_samples[i] for i in val_idx]
test_samples  = [all_samples[i] for i in test_idx]

print(f"Train: {len(train_samples)} pares → {len(train_samples)*2} imágenes")
print(f"Val:   {len(val_samples)} pares → {len(val_samples)*2} imágenes")
print(f"Test:  {len(test_samples)} pares → {len(test_samples)*2} imágenes")

# ======================
# SEQUENCE MULTICLASE
# ======================
class StegoSequence(tf.keras.utils.Sequence):
    def __init__(self, samples, batch_size, num_classes):
        self.samples     = samples
        self.batch       = batch_size
        self.num_classes = num_classes
        self.indices     = np.arange(len(self.samples))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.samples) * 2 / self.batch))

    def __getitem__(self, idx):
        half      = self.batch // 2
        batch_ids = self.indices[idx * half:(idx + 1) * half]
        X = np.zeros((len(batch_ids) * 2, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
        y = np.zeros((len(batch_ids) * 2, self.num_classes),       dtype=np.float32)
        j = 0
        for i in batch_ids:
            sample = self.samples[i]
            # COVER
            img = cv2.imread(sample["cover_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['cover_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(0, num_classes=self.num_classes)
            j += 1
            # STEGO
            img = cv2.imread(sample["stego_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['stego_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(sample["stego_label"], num_classes=self.num_classes)
            j += 1
        return X, y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_seq = StegoSequence(train_samples, BATCH_SIZE, NUM_CLASSES)
val_seq   = StegoSequence(val_samples,   BATCH_SIZE, NUM_CLASSES)
test_seq  = StegoSequence(test_samples,  BATCH_SIZE, NUM_CLASSES)

# ======================
# SANITY CHECK
# ======================
print("\n[CHECK] Probando un batch del Sequence...")
X_dbg, y_dbg = train_seq[0]
print("X batch shape:", X_dbg.shape)
print("y batch shape:", y_dbg.shape)
print("y sample (primeras 6):", y_dbg[:6])
print("[CHECK OK] Loader funcionando correctamente\n")

## Training image pairs(8000), Validation image pairs(1000), and Test image pairs(1000)

In [ ]:
COVER_DIR = "cover256gris"

# 5 familias de stego
STEGO_DIRS = {
    "Troyano":        "jslamegris256",
    "Gusanos":        "beagle",
    "Apt":            "stuxnet",
    "Puerta_trasera": "linuxchapros256gris",
    "Ransomware":     "thanos256gris",
}

# Clases: cover=0, familia1=1, ..., familia5=5
CLASS_NAMES = ["cover"] + list(STEGO_DIRS.keys())
NUM_CLASSES  = len(CLASS_NAMES)  # 6
print("Clases:", CLASS_NAMES)

IMG_SIZE   = 256
BATCH_SIZE = 32
EPOCHS     = 20
TEST_RATIO  = 0.10
VAL_RATIO   = 0.111   # fracción del train que va a validación
SEED       = 42

# ======================
# SRM
# ======================
srm_weights = np.load("filtro/SRM_Kernels1.npy")
biasSRM     = np.ones(30)

# ======================
# UTILS
# ======================
def get_id(f):
    return os.path.splitext(f)[0].split("_")[0]

# ======================
# DATA SPLIT (PAIR SAFE)
# ======================
cover_map = {get_id(f): f for f in os.listdir(COVER_DIR)}

all_samples = []
for label_idx, (family_name, stego_dir) in enumerate(STEGO_DIRS.items(), start=1):
    stego_map  = {get_id(f): f for f in os.listdir(stego_dir)}
    common_ids = sorted(set(cover_map.keys()) & set(stego_map.keys()))
    print(f"  [{family_name}] pares válidos: {len(common_ids)}")
    for img_id in common_ids:
        all_samples.append({
            "cover_path":  os.path.join(COVER_DIR, cover_map[img_id]),
            "stego_path":  os.path.join(stego_dir, stego_map[img_id]),
            "stego_label": label_idx
        })

print(f"\nTotal pares (todas las familias): {len(all_samples)}")

np.random.seed(SEED)
indices = np.arange(len(all_samples))
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_RATIO, random_state=SEED, shuffle=True
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=VAL_RATIO, random_state=SEED, shuffle=True
)

train_samples = [all_samples[i] for i in train_idx]
val_samples   = [all_samples[i] for i in val_idx]
test_samples  = [all_samples[i] for i in test_idx]

print(f"Train: {len(train_samples)} pares → {len(train_samples)*2} imágenes")
print(f"Val:   {len(val_samples)} pares → {len(val_samples)*2} imágenes")
print(f"Test:  {len(test_samples)} pares → {len(test_samples)*2} imágenes")

# ======================
# SEQUENCE MULTICLASE
# ======================
class StegoSequence(tf.keras.utils.Sequence):
    def __init__(self, samples, batch_size, num_classes):
        self.samples     = samples
        self.batch       = batch_size
        self.num_classes = num_classes
        self.indices     = np.arange(len(self.samples))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.samples) * 2 / self.batch))

    def __getitem__(self, idx):
        half      = self.batch // 2
        batch_ids = self.indices[idx * half:(idx + 1) * half]
        X = np.zeros((len(batch_ids) * 2, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
        y = np.zeros((len(batch_ids) * 2, self.num_classes),       dtype=np.float32)
        j = 0
        for i in batch_ids:
            sample = self.samples[i]
            # COVER
            img = cv2.imread(sample["cover_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['cover_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(0, num_classes=self.num_classes)
            j += 1
            # STEGO
            img = cv2.imread(sample["stego_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise FileNotFoundError(f"No se pudo leer: {sample['stego_path']}")
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[j, :, :, 0] = img.astype(np.float32) / 255.0
            y[j]           = tf.keras.utils.to_categorical(sample["stego_label"], num_classes=self.num_classes)
            j += 1
        return X, y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_seq = StegoSequence(train_samples, BATCH_SIZE, NUM_CLASSES)
val_seq   = StegoSequence(val_samples,   BATCH_SIZE, NUM_CLASSES)
test_seq  = StegoSequence(test_samples,  BATCH_SIZE, NUM_CLASSES)

# ======================
# SANITY CHECK
# ======================
print("\n[CHECK] Probando un batch del Sequence...")
X_dbg, y_dbg = train_seq[0]
print("X batch shape:", X_dbg.shape)
print("y batch shape:", y_dbg.shape)
print("y sample (primeras 6):", y_dbg[:6])
print("[CHECK OK] Loader funcionando correctamente\n")

## CNN name and algorithm 

In [ ]:
path_log_base = './logs/Zhu_Net/WOW'
path_img_base = './image/Zhu_Net/WOW'

## Training

In [ ]:
model= Zhu_Net() 
train(model, X_train, y_train, X_valid, y_valid, X_test, y_test, batch_size=32, epochs=100, model_name="Zhu_Net.....")

## Train and valid

In [ ]:
model= Zhu_Net() 
PATH_trained_models = ""
Final_Results_Test(model,PATH_trained_models)

## graphics

In [ ]:
graphics(history, AccTest, LossTest, PATH_trained_models, model_Name, lossTEST, lossTRAIN, lossVALID, accuracyTEST, accuracyTRAIN, accuracyVALID)

## Top

In [ ]:
top_models(AccTest, accuracyTRAIN, accuracyVALID)

Note: If you want to train the algorithm with S-UNIWARD 0.4 bpp, change "PATH04_WOW1" and  "base_name".